## T04 - Variable de control

#### Simulación de Procesos Financieros

José Armando Melchor Soto
 | 21/10/2026

---

### Libraries 

In [14]:
import numpy as np
import pandas as pd 
from scipy.stats import norm
import matplotlib.pyplot as plt

---

### Functions 

In [ ]:
def simular_rutas(initial_price, risk_free_rate, volatility, maturity, num_simulations, num_steps):
    """Simulate asset price paths using Geometric Brownian Motion."""
    dt = maturity / num_steps
    random_shocks = np.random.normal(0, 1, (num_simulations, num_steps))
    price_paths = np.zeros((num_simulations, num_steps + 1))
    price_paths[:, 0] = initial_price
    for step in range(num_steps):
        price_paths[:, step + 1] = price_paths[:, step] * np.exp(
            (risk_free_rate - volatility**2 / 2) * dt + volatility * np.sqrt(dt) * random_shocks[:, step])
    return price_paths


def calcular_promedios(price_paths):
    """Calculate arithmetic and geometric average prices for each path."""
    simulated_prices = price_paths[:, 1:]
    arithmetic_average = np.mean(simulated_prices, axis=1)
    geometric_average = np.exp(np.mean(np.log(simulated_prices), axis=1))
    return arithmetic_average, geometric_average


def calcular_payoffs(arithmetic_average, geometric_average, strike_price, risk_free_rate, maturity):
    """Calculate discounted payoffs for arithmetic and geometric Asian call options."""
    discount_factor = np.exp(-risk_free_rate * maturity)
    arithmetic_payoff = discount_factor * \
        np.maximum(arithmetic_average - strike_price, 0)
    geometric_payoff = discount_factor * \
        np.maximum(geometric_average - strike_price, 0)
    return arithmetic_payoff, geometric_payoff


def precio_geometrico_teorico(initial_price, strike_price, risk_free_rate, volatility, maturity, num_steps):
    """Calculate the theoretical price of a discretely monitored geometric Asian call."""
    mean_log_geometric = np.log(initial_price) + (risk_free_rate -
                                                  volatility**2 / 2) * maturity * (num_steps + 1) / (2 * num_steps)
    variance_log_geometric = volatility**2 * maturity * \
        (num_steps + 1) * (2 * num_steps + 1) / (6 * num_steps**2)
    std_log_geometric = np.sqrt(variance_log_geometric)
    d2 = (mean_log_geometric - np.log(strike_price)) / std_log_geometric
    d1 = d2 + std_log_geometric
    theoretical_price = np.exp(-risk_free_rate * maturity) * (np.exp(mean_log_geometric +
                                                                     variance_log_geometric / 2) * norm.cdf(d1) - strike_price * norm.cdf(d2))
    return theoretical_price


def calcular_beta(arithmetic_payoff, geometric_payoff):
    """Calculate the control variate coefficient."""
    covariance = np.cov(arithmetic_payoff, geometric_payoff, ddof=1)[0, 1]
    variance = np.var(geometric_payoff, ddof=1)
    return covariance / variance


def calcular_precios(arithmetic_payoff, geometric_payoff, theoretical_geometric_price, beta):
    """Estimate the arithmetic Asian option price with and without a control variate."""
    monte_carlo_price = np.mean(arithmetic_payoff)
    control_variate_price = monte_carlo_price - beta * \
        (np.mean(geometric_payoff) - theoretical_geometric_price)
    return monte_carlo_price, control_variate_price

---

### Parámetros

In [16]:
S0 = 100
K = 100
r = 0.05
sigma = 0.20
T = 1
n = 252       # Días de negociación
M = 100       # Rutas
dt = T / n

---

In [18]:
price_paths = simular_rutas(S0, r, sigma, T, M, n)
arithmetic_average, geometric_average = calcular_promedios(price_paths)
arithmetic_payoff, geometric_payoff = calcular_payoffs(arithmetic_average, geometric_average, K, r, T)
theoretical_geometric_price = precio_geometrico_teorico(S0, K, r, sigma, T, n)
beta = calcular_beta(arithmetic_payoff, geometric_payoff)
monte_carlo_price, control_variate_price = calcular_precios(arithmetic_payoff, geometric_payoff, theoretical_geometric_price, beta)

### Resultados

In [ ]:
print(f"Theoretical geometric Asian call price: {theoretical_geometric_price:.4f}")
print(f"Beta: {beta:.4f}")
print(f"Arithmetic Asian call price (Monte Carlo): {monte_carlo_price:.4f}")
print(f"Arithmetic Asian call price (Control Variate): {control_variate_price:.4f}")

Theoretical geometric Asian call price: 5.5655
Beta: 1.0288
Arithmetic Asian call price (Monte Carlo): 5.8886
Arithmetic Asian call price (Control Variate): 5.7593


### Conclusión

Al tomar -Z del Z simulado, se forzó un balance porque la distribución normal es simétrica. Este método ayuda a reducir la varianza al usar cada Z con su opuesto, lo que permite obtener una estimación más precisa. Por otro lado, al usar la variable de control, el precio Monte Carlo pasó de 5.8886 a 5.7593, acercándose al precio teórico de 5.5655 de la opción asiática geométrica.